# GAIL на MountainCar

Реализуем алгоритм GAIL на среде Mountain Car с экспертными данными из детерминированной стратегии.

In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
COMPUTE_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {COMPUTE_DEVICE}')

Using device: cuda


# Генерация экспертных данных

In [ ]:
from IPython import display
MAX_TIMESTEPS = 250
env = gym.wrappers.TimeLimit(
    gym.make('MountainCar-v0', render_mode='rgb_array'),
    max_episode_steps=MAX_TIMESTEPS + 1,
)
ACTION_MAP = {'left': 0, 'stop': 1, 'right': 2}

In [ ]:
def expert_controller(observation, time_step):
    position, velocity = observation

    if time_step <= 50:
        return ACTION_MAP['left']
    elif time_step <= 100:
        return ACTION_MAP['right']
    elif time_step <= 150:
        return ACTION_MAP['left']
    return ACTION_MAP['right']

In [ ]:
test_performance = []
for trial_num in range(30):
    obs, _ = env.reset()
    terminated = False
    cumulative_return = 0
    step_count = 0

    while not terminated:
        action = expert_controller(obs, step_count)
        obs, reward, done, truncated, _ = env.step(action)
        terminated = done or truncated
        cumulative_return += reward
        step_count += 1
        if step_count > 250:
            break

    test_performance.append(cumulative_return)

print(f'Средняя награда эксперта: {np.mean(test_performance):.2f}')

Средняя награда эксперта: -167.43


# GAIL

In [ ]:
expert_state_collection = []
expert_action_collection = []

for episode in range(75):
    episode_states, episode_actions = [], []
    obs, _ = env.reset()
    for step in range(MAX_TIMESTEPS):
        action = expert_controller(obs, step)
        episode_states.append(obs)
        episode_actions.append(action)
        next_obs, reward, done, truncated, _ = env.step(action)
        terminated = done or truncated
        obs = next_obs
        if terminated:
            break

    expert_state_collection.extend(episode_states)
    expert_action_collection.extend(episode_actions)

In [ ]:
expert_states = np.array(expert_state_collection)
expert_actions = np.array(expert_action_collection)

print(f'States shape: {expert_states.shape}')
print(f'Actions shape: {expert_actions.shape}')

States shape: (12181, 2)
Actions shape: (12181,)


In [ ]:
def add_temporal_features(state_sequence, max_steps=MAX_TIMESTEPS):
    extended_states = []
    for t, state in enumerate(state_sequence):
        phase = 2 * np.pi * t / max_steps
        time_features = np.array([np.sin(phase), np.cos(phase)])
        enhanced_state = np.concatenate([state, time_features])
        extended_states.append(enhanced_state)
    return np.array(extended_states)

expert_states_augmented = add_temporal_features(expert_states)

STATE_DIMENSION = 4
ACTION_DIMENSION = 3

expert_states_tensor = torch.tensor(expert_states_augmented, dtype=torch.float32, device=COMPUTE_DEVICE)
expert_actions_tensor = torch.tensor(expert_actions, dtype=torch.long, device=COMPUTE_DEVICE)

print(f"Expert shapes: {expert_states_tensor.shape}")

Expert shapes: obs=torch.Size([12181, 4]), acts=torch.Size([12181])
obs shape: (12181, 4)
acts shape: (12181,)


In [ ]:
class ActionPolicy(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, observation):
        logits = self.feature_extractor(observation)
        return Categorical(logits=logits)

    def sample_action(self, observation):
        distribution = self.forward(observation)
        return distribution.sample().item()

In [ ]:
class StateActionDiscriminator(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.discriminator_net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, state, action):
        action_encoding = F.one_hot(action, num_classes=3).float()
        combined = torch.cat([state, action_encoding], dim=1)
        return self.discriminator_net(combined)

In [ ]:
class EpisodeBuffer:
    def __init__(self):
        self.states, self.actions, self.rewards = [], [], []

    def add_transition(self, state, action, reward):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)

    def get_data(self):
        return (
            torch.tensor(np.array(self.states), dtype=torch.float32, device=COMPUTE_DEVICE),
            torch.tensor(np.array(self.actions), dtype=torch.long, device=COMPUTE_DEVICE),
            torch.tensor(np.array(self.rewards), dtype=torch.float32, device=COMPUTE_DEVICE)
        )

In [ ]:
training_environment = gym.make('MountainCar-v0')
policy_network = ActionPolicy(STATE_DIMENSION, ACTION_DIMENSION).to(COMPUTE_DEVICE)
discriminator_network = StateActionDiscriminator(STATE_DIMENSION, ACTION_DIMENSION).to(COMPUTE_DEVICE)

policy_optimizer = optim.Adam(policy_network.parameters(), lr=1e-3)
discriminator_optimizer = optim.Adam(discriminator_network.parameters(), lr=3e-4)

In [ ]:
expert_states_tensor = torch.tensor(expert_states_augmented, dtype=torch.float32, device=COMPUTE_DEVICE)
expert_actions_tensor = torch.tensor(expert_actions, dtype=torch.long, device=COMPUTE_DEVICE)

In [ ]:
EPOCHS = 5000
policy_loss_tracker = []
discriminator_loss_tracker = []

for epoch in range(EPOCHS):
    trajectory_buffer = EpisodeBuffer()
    obs, _ = training_environment.reset()
    episode_complete = False
    current_step = 0

    # Сбор траектории
    while not episode_complete:
        phase = 2 * np.pi * current_step / 250
        extended_obs = np.concatenate([obs, [np.sin(phase), np.cos(phase)]])

        obs_tensor = torch.tensor(extended_obs, dtype=torch.float32, device=COMPUTE_DEVICE).unsqueeze(0)
        action = policy_network.sample_action(obs_tensor)
        next_obs, _, done, truncated, _ = training_environment.step(action)
        episode_complete = done or truncated

        trajectory_buffer.add_transition(extended_obs, action, 0)
        obs = next_obs
        current_step += 1

        if current_step > 250:
            break

    agent_states, agent_actions, _ = trajectory_buffer.get_data()

    # Выбор случайной подвыборки экспертных данных
    sample_indices = np.random.choice(len(expert_states_tensor), len(agent_states), replace=False)
    expert_state_batch = expert_states_tensor[sample_indices]
    expert_action_batch = expert_actions_tensor[sample_indices]

    # Обновление дискриминатора
    for _ in range(2):
        discriminator_optimizer.zero_grad()
        
        expert_predictions = discriminator_network(expert_state_batch, expert_action_batch)
        agent_predictions = discriminator_network(agent_states, agent_actions)
        
        discriminator_loss = -torch.mean(torch.log(expert_predictions + 1e-8)) - torch.mean(torch.log(1 - agent_predictions + 1e-8))
        discriminator_loss.backward()
        discriminator_optimizer.step()

    # Вычисление наград для агента
    with torch.no_grad():
        agent_rewards = -torch.log(1 - discriminator_network(agent_states, agent_actions) + 1e-8).squeeze()
    
    # Обновление политики
    policy_optimizer.zero_grad()
    action_distribution = policy_network(agent_states)
    log_probs = action_distribution.log_prob(agent_actions)
    policy_loss = -(log_probs * agent_rewards).mean()
    policy_loss.backward()
    policy_optimizer.step()

    # Сохранение метрик
    policy_loss_tracker.append(policy_loss.item())
    discriminator_loss_tracker.append(discriminator_loss.item())

    # Вывод прогресса
    if epoch % 100 == 0:
        print(f'Epoch {epoch}: Policy Loss {policy_loss.item():.4f}, Disc Loss {discriminator_loss.item():.4f}')

Epoch 0: Policy Loss 0.8005, Disc Loss 1.3936
Epoch 100: Policy Loss 0.6357, Disc Loss 1.3196
Epoch 200: Policy Loss 0.4730, Disc Loss 1.2805
Epoch 300: Policy Loss 0.4232, Disc Loss 1.2218
Epoch 400: Policy Loss 0.3884, Disc Loss 1.1842
Epoch 500: Policy Loss 0.3750, Disc Loss 1.0894
Epoch 600: Policy Loss 0.3198, Disc Loss 1.0402
Epoch 700: Policy Loss 0.3626, Disc Loss 1.0017
Epoch 800: Policy Loss 0.2117, Disc Loss 1.1101
Epoch 900: Policy Loss 0.3513, Disc Loss 1.0000
Epoch 1000: Policy Loss 0.2140, Disc Loss 1.0312
Epoch 1100: Policy Loss 0.2904, Disc Loss 0.9137
Epoch 1200: Policy Loss 0.2465, Disc Loss 0.8153
Epoch 1300: Policy Loss 0.1997, Disc Loss 0.9221
Epoch 1400: Policy Loss 0.3860, Disc Loss 1.1607
Epoch 1500: Policy Loss 0.3055, Disc Loss 0.9461
Epoch 1600: Policy Loss 0.1852, Disc Loss 0.8908
Epoch 1700: Policy Loss 0.2341, Disc Loss 0.8709
Epoch 1800: Policy Loss 0.2025, Disc Loss 0.7678
Epoch 1900: Policy Loss 0.1704, Disc Loss 0.8963
Epoch 2000: Policy Loss 0.1998, 

# Тестирование алгоритма

In [ ]:
evaluation_rewards = []

for eval_episode in range(30):
    obs, _ = training_environment.reset()
    episode_ended = False
    total_return = 0
    step_counter = 0

    while not episode_ended:
        phase = 2 * np.pi * step_counter / 250
        augmented_observation = np.concatenate([obs, [np.sin(phase), np.cos(phase)]])

        state_tensor = torch.tensor(augmented_observation, dtype=torch.float32, device=COMPUTE_DEVICE).unsqueeze(0)
        selected_action = policy_network.sample_action(state_tensor)
        next_obs, reward, terminated, truncated, _ = training_environment.step(selected_action)
        episode_ended = terminated or truncated

        obs = next_obs
        total_return += reward
        step_counter += 1

        if step_counter > 250:
            break

    evaluation_rewards.append(total_return)
    print(f'Episode {eval_episode + 1}: Total Reward = {total_return}')

training_environment.close()

# Статистика результатов
print(f'Средняя награда: {np.mean(evaluation_rewards):.2f}')
print(f'Максимальная награда: {np.max(evaluation_rewards):.2f}')
print(f'Минимальная награда: {np.min(evaluation_rewards):.2f}')

Episode 1: Total Reward = -200.0
Episode 2: Total Reward = -200.0
Episode 3: Total Reward = -200.0
Episode 4: Total Reward = -200.0
Episode 5: Total Reward = -200.0
Episode 6: Total Reward = -200.0
Episode 7: Total Reward = -200.0
Episode 8: Total Reward = -200.0
Episode 9: Total Reward = -200.0
Episode 10: Total Reward = -200.0
Episode 11: Total Reward = -200.0
Episode 12: Total Reward = -200.0
Episode 13: Total Reward = -200.0
Episode 14: Total Reward = -200.0
Episode 15: Total Reward = -200.0
Episode 16: Total Reward = -200.0
Episode 17: Total Reward = -200.0
Episode 18: Total Reward = -186.0
Episode 19: Total Reward = -200.0
Episode 20: Total Reward = -200.0
Episode 21: Total Reward = -200.0
Episode 22: Total Reward = -200.0
Episode 23: Total Reward = -200.0
Episode 24: Total Reward = -200.0
Episode 25: Total Reward = -175.0
Episode 26: Total Reward = -175.0
Episode 27: Total Reward = -200.0
Episode 28: Total Reward = -200.0
Episode 29: Total Reward = -200.0
Episode 30: Total Rewar